# 43. Ensembles and advanced evaluation
Voting combines model predictions. Stacking trains a meta-model using predictions from other models, which must be generated without allowing each training example to be predicted by a base model that fitted it. Nested cross-validation separates hyperparameter selection in inner folds from evaluation in outer folds.

Use built-in stacking support for out-of-fold meta-features rather than training a meta-model on overfitted base predictions. Compare an ensemble with its strongest individual model. Nested evaluation estimates the selection procedure and costs many fits. Similar component errors can limit ensemble gains. A final reserved test can still be useful after selecting a process.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
X, y = load_breast_cancer(return_X_y=True)
X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
from sklearn.ensemble import VotingClassifier, RandomForestClassifier
from sklearn.model_selection import cross_val_score
ensemble = VotingClassifier(estimators=[("linear", model), ("forest", RandomForestClassifier(n_estimators=25, random_state=42))], voting="soft") # Average class probabilities.
scores = cross_val_score(ensemble, X_dev, y_dev, cv=3, scoring="accuracy") # Refit components inside each fold.
print("Ensemble mean:", round(scores.mean(), 3)) # Compare against individual models separately.
assert len(scores) == 3 # Check fold count.

## Hard voting, soft voting, AdaBoost, and stacking
Hard voting combines class decisions; soft voting combines probabilities and is sensitive to their calibration. AdaBoost sequentially reweights training emphasis, unlike independently bagged trees. Stacking trains a final estimator on out-of-fold predictions from base learners. passthrough=True additionally supplies original features to the final estimator; scale them there when needed. If preprocessing sits outside the stacking cross-fit, the meta-features may see preprocessing learned using its validation rows. Keep learned preprocessing inside each base pipeline, and keep final-estimator preprocessing within its own training workflow.

In [ ]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import VotingClassifier, StackingClassifier, AdaBoostClassifier
X, y = load_wine(return_X_y=True)
Xdev, Xtest, ydev, ytest = train_test_split(X, y, stratify=y, random_state=42) # Keep final-test examples reserved.
base = [("linear", make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))), ("tree", DecisionTreeClassifier(max_depth=3, random_state=42))]
inner = StratifiedKFold(3, shuffle=True, random_state=1)
outer = StratifiedKFold(3, shuffle=True, random_state=2)
models = {"hard vote": VotingClassifier(base, voting="hard"), "soft vote": VotingClassifier(base, voting="soft"), "AdaBoost": AdaBoostClassifier(n_estimators=25, random_state=42), "stacking": StackingClassifier(base, final_estimator=make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)), cv=inner, passthrough=True)} # Separate base and meta-fitting boundaries.
for name, model in models.items():
    score = cross_val_score(model, Xdev, ydev, cv=outer, scoring="accuracy") # Compare full unfitted workflows.
    print(name, round(score.mean(), 3))
assert len(score) == 3

### Reading the result
The outer scores compare complete workflows; stacking’s internal folds create meta-training features without fitting a row’s base predictor on that row. Raw features also reach the scaled final pipeline because passthrough is true. Soft voting does not automatically improve calibration, and the strongest individual baseline still needs comparison.

### Transfer exercise
Compare all four ensembles with the individual base estimators. Repeat stacking with passthrough=False and report cost alongside score.

## Your turn
Compare the ensemble and its components on identical folds. Sketch which data each stage of nested cross-validation may see.

Try the task before opening the solution on the course website.